In [ ]:
%pip install -U langchain langchain-groq langchain-huggingface langchain-chroma langchain-community  langgraph wikipedia sentence-transformers

In [5]:
import os
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.tools import create_retriever_tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langgraph.prebuilt import create_react_agent

# 1. API Key
os.environ["GROQ_API_KEY"] = "api_key"

# ==========================================
#  Tool 1: RAG (Local Document Database)
# ==========================================
# For simplicity in testing, instead of loading a PDF, we create a few lines of mock text as company data.
company_data = [
    Document(page_content="The CEO of TechNova company is Dr. Ali Rezaei. The company was founded in 2020."),
    Document(page_content="TechNova's main product is an AI-based email management system called SmartInbox."),
    Document(page_content="TechNova's main office is located in Tehran, Iran.")
]

# Create vector database (RAG)
print("Loading embeddings and creating local vector store...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(company_data, embeddings)
retriever = vectorstore.as_retriever()

# Convert RAG into an agent tool
rag_tool = create_retriever_tool(
    retriever,
    name="technova_database",
    description="Use this tool to answer any questions about TechNova company, its products, founders, or internal data."
)

# ==========================================
#  Tool 2: Web Search (Wikipedia)
# ==========================================
# Configure Wikipedia to return a summary of the best result
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1500)
web_tool = WikipediaQueryRun(api_wrapper=api_wrapper, description="Use this tool to search the internet/Wikipedia for general knowledge, public people, history, or science.")

# ==========================================
#  Create Agent
# ==========================================
tools = [rag_tool, web_tool]

# Using a fast and standard model
llm = ChatGroq(temperature=0, model_name="openai/gpt-oss-120b")
agent = create_react_agent(llm, tools)

print(" Agentic RAG is Ready! It can search both local files and the web.")

Loading embeddings and creating local vector store...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Agentic RAG is Ready! It can search both local files and the web.


/tmp/ipykernel_183/2232804166.py:51: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


In [6]:
# --- Test 1: Question about internal company data (Should use RAG) ---
question_1 = "محصول اصلی شرکت TechNova چیست؟"
print(f" Question 1: {question_1}")

response_1 = agent.invoke({"messages": [("user", question_1)]})
print(" Answer 1:")
print(response_1["messages"][-1].content)
print("-" * 50)

# --- Test 2: Question about general world knowledge (Should use Wikipedia) ---
question_2 = "پایتخت کشور ژاپن کجاست و چه جمعیتی دارد؟"
print(f" Question 2: {question_2}")

response_2 = agent.invoke({"messages": [("user", question_2)]})
print(" Answer 2:")
print(response_2["messages"][-1].content)

 Question 1: محصول اصلی شرکت TechNova چیست؟
 Answer 1:
محصول اصلی شرکت TechNova، سامانهٔ مدیریت ایمیل مبتنی بر هوش مصنوعی به نام **SmartInbox** است.
--------------------------------------------------
 Question 2: پایتخت کشور ژاپن کجاست و چه جمعیتی دارد؟
 Answer 2:
پایتخت ژاپن **توکیو** (Tokyo) است.  

- **موقعیت جغرافیایی:** توکیو در ناحیه کانتو (Kantō) در جزیرهٔ اصلی ژاپن، هونشو، و در سرتاسر خلیج توکیو (Tokyo Bay) قرار دارد. این شهر در شرق کشور و نزدیک به سواحل اقیانوس آرام است.  

- **جمعیت:**  
  - جمعیت **شهر توکیو (شهر‑محیطی یا ۲۳ بخش ویژه)** حدود **۱۴ میلیون نفر** (آمار ۲۰۲۳) است.  
  - اگر ناحیهٔ کل متروپولیتن یا «منطقهٔ بزرگ توکیو» را در نظر بگیریم، جمعیت به حدود **۳۷ میلیون نفر** می‌رسد (برآورد سازمان ملل برای سال ۲۰۲۶).  

بنابراین، توکیو نه تنها مرکز سیاسی ژاپن است، بلکه یکی از پرجمعیت‌ترین شهرهای جهان نیز محسوب می‌شود.
